In [16]:
from pyomo.environ import *
from pyomo.contrib.piecewise import PiecewiseLinearFunction
from pyomo.core.base import TransformationFactory
from pyomo.opt import SolverStatus, TerminationCondition
import pyomo.environ as pyo
import pyomo.dae as dae
from dataclasses import dataclass, field
from typing import Callable, List, Dict, Tuple
import numpy as np
import math
import matplotlib.pyplot as plt
import bisect
import itertools as it
#from pyomo.environ import ConcreteModel, Var, Constraint, Expression

In [17]:
# ====================== Imports ======================
from pyomo.environ import *
from pyomo.contrib.piecewise import PiecewiseLinearFunction
from pyomo.core.base import TransformationFactory
from pyomo.opt import SolverStatus, TerminationCondition
import pyomo.environ as pyo
from typing import List, Tuple
import numpy as np
import math
import itertools as it

In [18]:
# ====================== 1) 代数离散版 PID（等价于 DAE + BACKWARD） ======================
def build_pid_model_fd(
    T=10, h=0.1, scen=None, weights=(1.0, 0.01),
    bounds={"x":(-20,20), "u":(-20,20), "Kp":(0,100), "Ki":(0,1000), "Kd":(0,1000)},
):
    """
    代数离散 (BACKWARD/隐式欧拉) 的 PID 单场景模型，与 DAE+finite_difference(BACKWARD) 等价：
      (x_t - x_{t-1})/h = (-x_t + Ku*u_t + d_t)/tau
      I_t = I_{t-1} + h*e_t
      u_t = Kp*e_t + Ki*I_t + Kd*(e_t - e_{t-1})/h  (t=0 时导数项取 0)
      e_t = sp_t - x_t
      目标：sum_t h*(w_e*e_t^2 + w_u*u_t^2)
    """
    assert scen is not None, "scen 不能为空"
    Ku, tau, d_seq, sp_seq = scen["Ku"], scen["tau"], scen["d"], scen["sp"]
    assert len(d_seq)==T+1 and len(sp_seq)==T+1, "d_seq / sp_seq 长度应为 T+1"

    m = pyo.ConcreteModel()
    m.T  = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)  # 1..T

    # 一阶段增益（有界，便于后续 big-M / GDP 变换安全）
    m.Kp = pyo.Var(bounds=bounds["Kp"])
    m.Ki = pyo.Var(bounds=bounds["Ki"])
    m.Kd = pyo.Var(bounds=bounds["Kd"])

    # 轨迹变量
    m.x = pyo.Var(m.T, bounds=bounds["x"])
    m.e = pyo.Var(m.T)
    m.u = pyo.Var(m.T, bounds=bounds["u"])
    m.I = pyo.Var(m.T)

    # 已知序列（参数）
    m.d  = pyo.Param(m.T, initialize={t: float(d_seq[t]) for t in m.T}, mutable=False)
    m.sp = pyo.Param(m.T, initialize={t: float(sp_seq[t]) for t in m.T}, mutable=False)

    # e = sp - x
    def _e_def(m, t): return m.e[t] == m.sp[t] - m.x[t]
    m.e_def = pyo.Constraint(m.T, rule=_e_def)

    # 隐式欧拉的状态方程
    def _x_dyn(m, t):
        return (m.x[t] - m.x[t-1]) / h == (-m.x[t] + Ku*m.u[t] + m.d[t]) / tau
    m.x_dyn = pyo.Constraint(m.Tm, rule=_x_dyn)

    # I 的累积
    def _I_acc(m, t): return m.I[t] == m.I[t-1] + h * m.e[t]
    m.I_acc = pyo.Constraint(m.Tm, rule=_I_acc)

    # 初值
    m.x0 = pyo.Constraint(expr=m.x[0] == 0.0)
    m.I0 = pyo.Constraint(expr=m.I[0] == 0.0)

    # e'(t) 的后向差分
    def dedt(m, t): 
        return 0.0 if t == 0 else (m.e[t] - m.e[t-1]) / h

    # PID 控制律
    def _pid(m, t):
        return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t] + m.Kd*dedt(m, t)
    m.pid = pyo.Constraint(m.T, rule=_pid)

    # 目标表达式（保留 expr，外层可自由加/减 As）
    w_e, w_u = weights
    m.obj_expr = pyo.Expression(expr=sum(h*(w_e*m.e[t]**2 + w_u*m.u[t]**2) for t in m.T))

    return m, [m.Kp, m.Ki, m.Kd]


# ====================== 2) 实用函数：清理组件 / 角点生成 ======================
def del_components(model, base_name="pw"):
    """
    清理上轮迭代留下的 piecewise 相关组件，避免撞名或污染。
    """
    for nm in ['obj', 'As', 'pw', 'pw_fun', 'pw_As', 'pw_link',
               f'{base_name}_blk', f'{base_name}_fun', f'{base_name}_fun_cc']:
        if hasattr(model, nm):
            model.del_component(nm)

    # 清理所有 *_fun_cc 残留 block（变换生成）
    to_delete = []
    for comp in list(model.component_objects(ctype=pyo.Block, descend_into=False)):
        if comp.name.endswith('_fun_cc'):
            to_delete.append(comp.name)
    for nm in to_delete:
        model.del_component(nm)


def corners_from_bounds(first_stg_vars: List[pyo.Var]):
    """给一组有界的一阶段变量生成 box 角点（每维取 lb/ub）"""
    bounds = []
    for y in first_stg_vars:
        lb, ub = y.lb, y.ub
        if lb is None or ub is None:
            raise ValueError(f"{y.name} 缺少上下界，无法生成角点")
        bounds.append((float(lb), float(ub)))
    return list(it.product(*[(lb, ub) for (lb, ub) in bounds]))


# ====================== 3) 分段近似：封装到独立 Block，单独变换 ======================
def add_nd_piecewise(
    model,
    first_stg_vars: List[pyo.Var],   # 自变量（如 [Kp, Ki, Kd]）
    points: List[Tuple[float, ...]], # 节点：[(...), (...), ...]
    values,                          # 节点值：与 points 对齐的 list 或 {point_tuple: val}
    name="pw",
    relation="==",                   # '==', '>='(下界), '<='(上界)
    round_ndigits=12,
):
    if len(points) == 0:
        raise ValueError("points 不能为空")
    N = len(first_stg_vars)
    for pt in points:
        if len(pt) != N:
            raise ValueError(f"points 维度与变量数不一致: {pt} vs {N}")

    # 清理旧同名组件
    for nm in [f"{name}_blk", f"{name}_fun", f"{name}_fun_cc"]:
        if hasattr(model, nm):
            model.del_component(nm)

    # 统一键
    def keyize(coords):
        return tuple(round(float(c), round_ndigits) for c in coords)
    norm_points = [keyize(pt) for pt in points]

    # 整理 value 表
    if isinstance(values, dict):
        table = {keyize(k): float(v) for k, v in values.items()}
        miss = [pt for pt in norm_points if pt not in table]
        if miss:
            raise KeyError(f"values 缺少这些点的取值: {miss[:5]}{' ...' if len(miss)>5 else ''}")
    else:
        if len(values) != len(points):
            raise ValueError("values 长度应与 points 数量一致")
        table = {pt: float(v) for pt, v in zip(norm_points, values)}

    def _f_from_table(*coords): return table[keyize(coords)]

    # 独立小 Block：只装 pw / As / link
    b = pyo.Block(concrete=True)
    model.add_component(f"{name}_blk", b)

    pw = PiecewiseLinearFunction(points=norm_points, function=_f_from_table, name=f"{name}_fun")
    b.add_component(pw.name, pw)

    pw_expr = pw(*first_stg_vars)

    # 给 As 加界，避免 -inf
    b.As = pyo.Var(bounds=(-1e9, 1e9))
    if relation == "==":
        b.link = pyo.Constraint(expr=b.As == pw_expr)
    elif relation == ">=":
        b.link = pyo.Constraint(expr=b.As >= pw_expr)
    elif relation == "<=":
        b.link = pyo.Constraint(expr=b.As <= pw_expr)
    else:
        raise ValueError("relation 只能是 '==', '>=', '<='")

    # 关键：只对 b 做变换（避免扫到外层其他组件）
    TransformationFactory('contrib.piecewise.convex_combination').apply_to(b)

    return b.As, pw


# ====================== 4) 克隆模板+取变量 ======================
def clone_and_get_vars(m_old, first_stage_vars):
    m_new = m_old.clone()
    first_stage_vars_new = []
    for v in first_stage_vars:
        v_new = m_new.find_component(v.name)
        if v_new is None:
            raise KeyError(f"在新模型里找不到变量 '{v.name}'")
        first_stage_vars_new.append(v_new)
    return m_new, first_stage_vars_new


# ====================== 5) 评估 v(y) 工具 ======================
def evaluate_Q_at(model, first_stg_vars, first_stg_vals, solver):
    """
    给定一阶段 y，最小化 obj_expr，返回 v(y)。
    （临时挂 obj，再删除；同时 fix/unfix 一阶段变量）
    """
    del_components(model)
    for u, v in zip(first_stg_vars, first_stg_vals):
        u.fix(value(v))
    model.obj = Objective(expr=model.obj_expr, sense=minimize)
    results = solver.solve(model, tee=False)

    status_ok = (results.solver.status == SolverStatus.ok)
    term_ok = (results.solver.termination_condition == TerminationCondition.optimal)
    if not (status_ok and term_ok):
        raise RuntimeError(f"evaluate_Q_at y={first_stg_vals} not optimal: "
                           f"status={results.solver.status}, term={results.solver.termination_condition}")

    v_opt = value(model.obj_expr)
    model.del_component('obj')
    for u in first_stg_vars:
        u.unfix()
    return v_opt


# ====================== 6) 主流程：nc_underest ======================
def nc_underest(model_list, first_stg_vars_list, m_tmpl_list, target_nodes,
                picture_shown=False, v_list=False, tolerance=1e-8):
    """
    - model_list: 每个场景的代数离散模型（由 build_pid_model_fd 构建）
    - first_stg_vars_list: 对应每个模型的一阶段变量列表（如 [Kp,Ki,Kd]）
    - m_tmpl_list: [模板模型, 模板模型的一阶段变量列表]（用于 sum/As_sum）
    - target_nodes: 目标节点数（>= 角点数）
    """
    N = len(model_list)
    solver = SolverFactory('gurobi')  # 你也可以外部传入或设置选项

    # 初始节点：角点
    first_stg_nodes = corners_from_bounds(first_stg_vars_list[0])
    as_nodes_list = [[] for _ in range(N)]

    # 评估每个场景在角点的 v(y)
    for i in range(N):
        as_nodes_list[i].extend(
            evaluate_Q_at(model_list[i], first_stg_vars_list[i], node, solver) for node in first_stg_nodes
        )
    print('corner nodes are ', first_stg_nodes)
    print('as_nodes_list are ', as_nodes_list)

    if target_nodes <= len(first_stg_nodes):
        print('target_nodes number should be larger than ', len(first_stg_nodes))
        return

    As_min_list, add_node_history, k_list = [], [], []
    ms_list = [None]*N
    new_nodes_list = [None]*N

    print('Start from ', len(first_stg_nodes), ' nodes')
    print('The goal is to get ', target_nodes, ' nodes')

    for k in range(len(first_stg_nodes)+1, target_nodes+1):
        print('##################################################')
        print('##################################################')
        print('Start adding node ', k)
        k_list.append(k)

        # 逐场景：构造 pw≈Q_i(y)，解 min (obj_expr - As) 得 ms_i 和候选新点
        for i in range(N):
            print('\nSolving scenario ', i)
            del_components(model_list[i])
            As_i, _ = add_nd_piecewise(
                model_list[i], first_stg_vars_list[i], first_stg_nodes, as_nodes_list[i],
                name="pw", relation="=="
            )
            model_list[i].obj = Objective(expr=model_list[i].obj_expr - As_i, sense=minimize)
            results = solver.solve(model_list[i], tee=False)
            if (results.solver.status != SolverStatus.ok) or \
               (results.solver.termination_condition != TerminationCondition.optimal):
                print("⚠ scenario solve may have issues")

            ms_list[i] = value(model_list[i].obj)
            new_nodes_list[i] = tuple(value(v) for v in first_stg_vars_list[i])
            print('new node is ', new_nodes_list[i])
            print('ms is ', ms_list[i])

        # 组合 As_sum：在模板模型上拟合 ∑_i Q_i 的节点值，解 min As_sum
        arr = np.array(as_nodes_list, dtype=float, ndmin=2)
        assum_nodes = arr.sum(axis=0)

        model_sum, model_sum_first_stg_vars = clone_and_get_vars(m_tmpl_list[0], m_tmpl_list[1])
        del_components(model_sum)
        As_sum, pw_sum = add_nd_piecewise(
            model_sum, model_sum_first_stg_vars, first_stg_nodes, assum_nodes,
            name="pw", relation="=="
        )
        model_sum.obj = Objective(expr=As_sum, sense=minimize)
        results = solver.solve(model_sum, tee=False)
        if not ((results.solver.status == SolverStatus.ok) and
                (results.solver.termination_condition == TerminationCondition.optimal)):
            print("Sum model doesn't get solved normally")

        # 更稳妥：直接取目标值，而不是 lower_bound 字段
        As_min = value(model_sum.obj)
        print(f'As_min at possible new node is {As_min}')
        node_star = tuple(value(v) for v in model_sum_first_stg_vars)

        # 如果 node_star 无效/重复，退化为“中心点”
        if (node_star is None) or (node_star in first_stg_nodes):
            avg = []
            for j in range(len(first_stg_nodes[0])):
                comp_vals = [node[j] for node in first_stg_nodes]
                avg.append(sum(comp_vals) / len(comp_vals))
            node_star = tuple(avg)
            As_min = value(pw_sum(*node_star))

        # 真实误差：|As_sum(y*) - ∑ v_i(y*)|
        real_at_star = 0.0
        for i in range(N):
            real_at_star += evaluate_Q_at(model_list[i], first_stg_vars_list[i], node_star, solver)
        err_star = abs(As_min - real_at_star)

        sum_ms = float(sum(ms_i for ms_i in ms_list))
        print('Sum *****************************************')
        print('error at y_star is ', err_star)
        print('y_star is ', node_star)
        print('ms_list and sum_ms is ', ms_list, sum_ms)

        # 选新点：如果误差主导，就用 y*；否则取 ms 最小对应的新点
        if err_star > abs(sum_ms):
            new_node = node_star
            print('new node chosen from error')
        else:
            min_index = int(np.argmin(ms_list))
            new_node = new_nodes_list[min_index]
            print('new node chosen from ms')

        As_min_list.append(As_min + sum_ms)
        add_node_history.append(new_node)
        print('new node is', new_node)
        print('Current As_min is', As_min_list[-1])
        print('')

        # 更新节点与各场景值
        first_stg_nodes.append(new_node)
        for i in range(N):
            as_nodes_list[i].append(evaluate_Q_at(model_list[i], first_stg_vars_list[i], new_node, solver))

    # 结束后再解一遍 As_sum 取最终下界
    arr = np.array(as_nodes_list, dtype=float, ndmin=2)
    assum_nodes = arr.sum(axis=0)

    model_sum, model_sum_first_stg_vars = clone_and_get_vars(m_tmpl_list[0], m_tmpl_list[1])
    del_components(model_sum)
    As_sum, _ = add_nd_piecewise(
        model_sum, model_sum_first_stg_vars, first_stg_nodes, assum_nodes,
        name="pw", relation="=="
    )
    model_sum.obj = Objective(expr=As_sum, sense=minimize)
    results = solver.solve(model_sum, tee=False)
    if not ((results.solver.status == SolverStatus.ok) and
            (results.solver.termination_condition == TerminationCondition.optimal)):
        print("Sum model (final) doesn't get solved normally")

    output_lb = value(model_sum.obj) + sum(ms_list)
    node_final = tuple(value(v) for v in model_sum_first_stg_vars)
    print('lower bound is ', output_lb)
    print('node is ', node_final)

    return output_lb, first_stg_nodes, [k_list, As_min_list, add_node_history]


# ====================== 7) 示例：单场景跑一下 ======================
if __name__ == "__main__":
    T, h = 30, 0.1
    times = [t for t in range(T+1)]

    def step_sp(t): return 1.0 if t*h >= 0.5 else 0.0
    def make_d(amp): return [amp*math.sin(0.5*t*h) for t in times]

    scenarios = {
        1: {"prob": 0.4, "Ku": 3.0, "tau": 2.0, "d": make_d(0.2), "sp": [step_sp(t) for t in times]},
        # 如需多场景，按同格式增添：
        # 2: {"prob": 0.4, "Ku": 2.7, "tau": 1.8, "d": make_d(0.5), "sp": [step_sp(t) for t in times]},
        # 3: {"prob": 0.2, "Ku": 3.3, "tau": 2.2, "d": make_d(0.8), "sp": [step_sp(t) for t in times]},
    }

    model_list = []
    first_stg_vars_list = []
    for _, scen in scenarios.items():
        m, m_f_stg_vars = build_pid_model_fd(T=T, h=h, scen=scen)   # 注意：这里用 FD 版
        model_list.append(m)
        first_stg_vars_list.append(m_f_stg_vars)

    # sum 模型的模板（只需要一阶段变量容器即可）
    bounds={"x":(-20,20), "u":(-20,20), "Kp":(0,100), "Ki":(0,1000), "Kd":(0,1000)}
    m_tmpl = pyo.ConcreteModel()
    m_tmpl.Kp = pyo.Var(bounds=bounds["Kp"])
    m_tmpl.Ki = pyo.Var(bounds=bounds["Ki"])
    m_tmpl.Kd = pyo.Var(bounds=bounds["Kd"])
    m_tmpl_list = [m_tmpl, [m_tmpl.Kp, m_tmpl.Ki, m_tmpl.Kd]]

    # 运行 underest
    nc_underest(model_list, first_stg_vars_list, m_tmpl_list,
                target_nodes=10, picture_shown=False, v_list=False, tolerance=1e-8)


corner nodes are  [(0.0, 0.0, 0.0), (0.0, 0.0, 1000.0), (0.0, 1000.0, 0.0), (0.0, 1000.0, 1000.0), (100.0, 0.0, 0.0), (100.0, 0.0, 1000.0), (100.0, 1000.0, 0.0), (100.0, 1000.0, 1000.0)]
as_nodes_list are  [[2.312013591358772, 0.050724842025142165, 0.046382008806636635, 0.05073328930301407, 0.04546167267640338, 0.05072629135759718, 0.047944356480856605, 0.05073385265647712]]
Start from  8  nodes
The goal is to get  10  nodes
##################################################
##################################################
Start adding node  9

Solving scenario  0
new node is  (8.037722090352561e-08, 6.385681815005446, 6.385680598918182)
ms is  -2.243381281201294
As_min at possible new node is 0.04546167267640344
Sum *****************************************
error at y_star is  0.0025737010140090694
y_star is  (50.0, 500.0, 500.0)
ms_list and sum_ms is  [-2.243381281201294] -2.243381281201294
new node chosen from ms
new node is (8.037722090352561e-08, 6.385681815005446, 6.38568059891

In [9]:
import pyomo.environ as pyo

def build_joint_pid_model_fd(
    scenarios: dict,          # {sid: {"prob":..., "Ku":..., "tau":..., "d":[T+1], "sp":[T+1]}}
    T=30, h=0.1,
    weights=(1.0, 0.01),
    bounds={"x":(-20,20), "u":(-20,20), "Kp":(0,100), "Ki":(0,1000), "Kd":(0,1000)},
):
    """
    构建“确定性等价”联合模型：
      - 共享的一阶段变量: Kp, Ki, Kd
      - 每个场景一个 Block，内含离散(隐式欧拉)的轨迹与约束
      - 目标 = sum_s prob_s * obj_expr_s
    """
    m = pyo.ConcreteModel()
    w_e, w_u = weights

    # 共享增益
    m.Kp = pyo.Var(bounds=bounds["Kp"])
    m.Ki = pyo.Var(bounds=bounds["Ki"])
    m.Kd = pyo.Var(bounds=bounds["Kd"])

    # 场景索引
    sids = sorted(list(scenarios.keys()))
    m.S = pyo.Set(initialize=sids)

    # 时间索引
    m.T  = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)

    # 为每个场景建 block
    m.B = pyo.Block(m.S)

    for s in sids:
        data = scenarios[s]
        prob, Ku, tau = data["prob"], data["Ku"], data["tau"]
        d_seq, sp_seq = data["d"], data["sp"]
        assert len(d_seq)==T+1 and len(sp_seq)==T+1

        b = m.B[s]
        # 轨迹变量
        b.x = pyo.Var(m.T, bounds=bounds["x"])
        b.e = pyo.Var(m.T)
        b.u = pyo.Var(m.T, bounds=bounds["u"])
        b.I = pyo.Var(m.T)

        # 参数
        b.d  = pyo.Param(m.T, initialize={t: float(d_seq[t]) for t in m.T}, mutable=False)
        b.sp = pyo.Param(m.T, initialize={t: float(sp_seq[t]) for t in m.T}, mutable=False)

        # e = sp - x
        def _e_def(b, t): return b.e[t] == b.sp[t] - b.x[t]
        b.e_def = pyo.Constraint(m.T, rule=_e_def)

        # 隐式欧拉状态方程
        def _x_dyn(b, t):
            return (b.x[t] - b.x[t-1]) / h == (-b.x[t] + Ku*b.u[t] + b.d[t]) / tau
        b.x_dyn = pyo.Constraint(m.Tm, rule=_x_dyn)

        # I 的累积
        def _I_acc(b, t): return b.I[t] == b.I[t-1] + h * b.e[t]
        b.I_acc = pyo.Constraint(m.Tm, rule=_I_acc)

        # 初值
        b.x0 = pyo.Constraint(expr=b.x[0] == 0.0)
        b.I0 = pyo.Constraint(expr=b.I[0] == 0.0)

        # e'(t) 后向差分
        def dedt(b, t):
            return 0.0 if t == 0 else (b.e[t] - b.e[t-1]) / h

        # PID 控制律（注意：用父模型的 Kp/Ki/Kd）
        def _pid(b, t):
            return b.u[t] == m.Kp*b.e[t] + m.Ki*b.I[t] + m.Kd*dedt(b, t)
        b.pid = pyo.Constraint(m.T, rule=_pid)

        # 场景目标表达式（不立刻加权）
        b.obj_expr = pyo.Expression(expr=sum(h*(w_e*b.e[t]**2 + w_u*b.u[t]**2) for t in m.T))
        # 存一下概率
        b.prob = prob

    # 全局目标：各场景期望
    m.obj = pyo.Objective(
        expr=sum(m.B[s].prob * m.B[s].obj_expr for s in m.S),
        sense=pyo.minimize
    )
    return m


In [12]:
from pyomo.opt import SolverStatus, TerminationCondition

import random

def multistart_joint(model_builder, scenarios, T, h, weights, bounds,
                     n_starts=20, seed=0):
    random.seed(seed)
    best_val = float('inf')
    best_sol = None

    for k in range(n_starts):
        m = model_builder(scenarios, T=T, h=h, weights=weights, bounds=bounds)
        # 随机初始化 Kp,Ki,Kd（有界域内）
        for var, (lb, ub) in [(m.Kp, bounds["Kp"]),
                              (m.Ki, bounds["Ki"]),
                              (m.Kd, bounds["Kd"])]:
            var.set_value(lb + (ub-lb)*random.random())
        ok = solve_joint_local(m)
        if ok:
            val = pyo.value(m.obj)
            if val < best_val:
                best_val = val
                best_sol = (pyo.value(m.Kp), pyo.value(m.Ki), pyo.value(m.Kd))
    return best_val, best_sol



In [14]:
# 你的场景
T, h = 30, 0.1
times = [t for t in range(T+1)]
def step_sp(t): return 1.0 if t*h >= 0.5 else 0.0
def make_d(amp): return [amp*math.sin(0.5*t*h) for t in times]

scenarios = {
    1: {"prob": 0.4, "Ku": 3.0, "tau": 2.0, "d": make_d(0.2), "sp": [step_sp(t) for t in times]},
    # 若有多场景，按同格式加进来
    # 2: {...}, 3: {...}
}

bounds={"x":(-20,20), "u":(-20,20), "Kp":(0,100), "Ki":(0,1000), "Kd":(0,1000)}
weights=(1.0, 0.01)

# A) 单次 Ipopt（局部）
m_joint = build_joint_pid_model_fd(scenarios, T=T, h=h, weights=weights, bounds=bounds)
ok = solve_joint_local(m_joint)
if ok:
    exact_val_local = pyo.value(m_joint.obj)
    exact_K = (pyo.value(m_joint.Kp), pyo.value(m_joint.Ki), pyo.value(m_joint.Kd))
    print("[Local] objective =", exact_val_local, "K* =", exact_K)

# B) 多起点（更稳）
best_val, best_K = multistart_joint(
    build_joint_pid_model_fd, scenarios, T, h, weights, bounds,
    n_starts=30, seed=42
)
print("[Multistart] best objective =", best_val, "best K* =", best_K)


[Local] objective = 0.015466619144272232 K* = (19.37427510374788, 6.8817168033451965, 4.067262177588943e-06)
[Multistart] best objective = 0.015466619144272229 best K* = (19.37427510374892, 6.881716803344667, 4.067262046897481e-06)
